In [1]:
import os
import torch
from torchvision import transforms, models
from PIL import Image
import numpy as np
import torch_directml
from torch.utils.data import Dataset, DataLoader
import json
from collections import defaultdict

In [2]:
# Set device
dml = torch_directml.device()

# --- Paths and model ---
# UPDATE THIS PATH TO YOUR 126-CLASS MODEL CHECKPOINT
ckpt_path = r'D:\VSC FILES\testtrain\runs\efficientnet_b3_baseline-20251116-101749\best_efficientnet_b3.pth'

# Dataset root (should contain 126 class folders: 125 food + non_food)
root_dir = r'D:\VSC FILES\testtrain\splits_new_v2'

# Load checkpoint
ckpt = torch.load(ckpt_path, map_location='cpu')
classes = ckpt['classes']
num_classes = len(classes)

print(f"Loaded model with {num_classes} classes")
print(f"Classes: {', '.join(classes[:5])}...{classes[-1]}")

# Model
model = models.efficientnet_b3(weights=None)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, num_classes)
model.load_state_dict(ckpt['model_state'])
model = model.to(dml)
model.eval()

# Transforms (use eval transforms from training)
mean, std = ckpt['mean'], ckpt['std']
eval_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(ckpt['image_size']),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

print(f"\nModel loaded successfully!")
print(f"Image size: {ckpt['image_size']}")

C:\Users\JP\AppData\Local\Temp\ipykernel_23916\2108698308.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location='cpu')


Loaded model with 126 classes
Classes: adobong_pusit, apple, arroz_caldo, baby_back_ribs, baked_tahong...white_rice

Model loaded successfully!
Image size: 252

Model loaded successfully!
Image size: 252


In [3]:
class ImageFolderFlat(Dataset):
    """Dataset that loads all images from class folders recursively (train/val/test combined)"""
    def __init__(self, root, classes, transform=None):
        self.samples = []
        self.labels = []
        self.class_names = []  # Track actual class name for each sample
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        
        for cls in classes:
            cls_dir = os.path.join(root, cls)
            if not os.path.isdir(cls_dir):
                continue
            # Walk class directory recursively
            for dirpath, _, filenames in os.walk(cls_dir):
                for fn in filenames:
                    if fn.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                        p = os.path.join(dirpath, fn)
                        self.samples.append(p)
                        self.labels.append(self.class_to_idx[cls])
                        self.class_names.append(cls)
        
        self.transform = transform
        self.classes = classes
        print(f"Loaded {len(self.samples)} images from {len(classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img = Image.open(self.samples[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx], self.samples[idx], self.class_names[idx]

# Create dataset and dataloader
dataset = ImageFolderFlat(root_dir, classes, transform=eval_tfms)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

Loaded 53918 images from 126 classes


In [4]:
# Initialize metrics tracking
class_metrics = defaultdict(lambda: {
    'total': 0,
    'top1_correct': 0,
    'top3_70conf_correct': 0,
    'nonfood_in_top3': 0,  # How many times "non_food" appeared in top-3
})

# Overall metrics
total_images = 0
overall_top1_correct = 0
overall_top3_70conf_correct = 0

# Get non_food class index
nonfood_idx = classes.index('non_food') if 'non_food' in classes else -1

print("Running inference on all images...")
print(f"Non-food class index: {nonfood_idx}\n")

with torch.no_grad():
    for batch_idx, (imgs, labels, paths, class_names_batch) in enumerate(loader):
        imgs = imgs.to(dml)
        logits = model(imgs)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        labels_np = labels.numpy()
        
        for i in range(len(imgs)):
            total_images += 1
            gt_label = labels_np[i]
            gt_class = class_names_batch[i]
            
            # Get top-3 predictions
            top3_idx = probs[i].argsort()[-3:][::-1]
            top3_probs = probs[i][top3_idx]
            top3_classes = [classes[j] for j in top3_idx]
            
            # Top-1 accuracy
            top1_pred = top3_idx[0]
            class_metrics[gt_class]['total'] += 1
            
            if top1_pred == gt_label:
                class_metrics[gt_class]['top1_correct'] += 1
                overall_top1_correct += 1
            
            # Top-3 with ≥70% confidence
            if gt_label in top3_idx:
                gt_pos = list(top3_idx).index(gt_label)
                if top3_probs[gt_pos] >= 0.7:
                    class_metrics[gt_class]['top3_70conf_correct'] += 1
                    overall_top3_70conf_correct += 1
            
            # Check if "non_food" is in top-3 (for food classes only)
            if gt_class != 'non_food' and nonfood_idx in top3_idx:
                class_metrics[gt_class]['nonfood_in_top3'] += 1
        
        # Progress update
        if (batch_idx + 1) % 100 == 0:
            print(f"Processed {total_images} images...")

print(f"\nInference complete! Processed {total_images} images.")

Running inference on all images...
Non-food class index: 75

Processed 3200 images...
Processed 3200 images...
Processed 6400 images...
Processed 6400 images...
Processed 9600 images...
Processed 9600 images...
Processed 12800 images...
Processed 12800 images...
Processed 16000 images...
Processed 16000 images...
Processed 19200 images...
Processed 19200 images...
Processed 22400 images...
Processed 22400 images...
Processed 25600 images...
Processed 25600 images...
Processed 28800 images...
Processed 28800 images...
Processed 32000 images...
Processed 32000 images...
Processed 35200 images...
Processed 35200 images...
Processed 38400 images...
Processed 38400 images...
Processed 41600 images...
Processed 41600 images...
Processed 44800 images...
Processed 44800 images...
Processed 48000 images...
Processed 48000 images...
Processed 51200 images...
Processed 51200 images...

Inference complete! Processed 53918 images.

Inference complete! Processed 53918 images.


In [5]:
# Calculate overall metrics
overall_top1_acc = (overall_top1_correct / total_images * 100) if total_images else 0
overall_top3_70conf_acc = (overall_top3_70conf_correct / total_images * 100) if total_images else 0

print(f"\n{'='*80}")
print(f"126-CLASS MODEL INFERENCE RESULTS")
print(f"{'='*80}")
print(f"Total images tested: {total_images}")
print(f"Overall Top-1 Accuracy: {overall_top1_acc:.2f}% ({overall_top1_correct}/{total_images})")
print(f"Overall Top-3 Accuracy (≥70% conf): {overall_top3_70conf_acc:.2f}% ({overall_top3_70conf_correct}/{total_images})")
print(f"{'='*80}\n")

# Calculate per-class metrics
per_class_results = []
for cls_name in sorted(class_metrics.keys()):
    metrics = class_metrics[cls_name]
    total = metrics['total']
    if total == 0:
        continue
    
    top1_acc = metrics['top1_correct'] / total * 100
    top3_70conf_acc = metrics['top3_70conf_correct'] / total * 100
    nonfood_contamination_rate = metrics['nonfood_in_top3'] / total * 100
    
    per_class_results.append({
        'class': cls_name,
        'total_images': total,
        'top1_correct': metrics['top1_correct'],
        'top1_accuracy': top1_acc,
        'top3_70conf_correct': metrics['top3_70conf_correct'],
        'top3_70conf_accuracy': top3_70conf_acc,
        'nonfood_in_top3_count': metrics['nonfood_in_top3'],
        'nonfood_contamination_rate': nonfood_contamination_rate
    })

# Sort by top1 accuracy (worst first)
per_class_results_sorted = sorted(per_class_results, key=lambda x: x['top1_accuracy'])

print("Bottom 10 classes by Top-1 Accuracy:")
for i, result in enumerate(per_class_results_sorted[:10], 1):
    print(f"  {i}. {result['class']}: {result['top1_accuracy']:.2f}% ({result['top1_correct']}/{result['total_images']})")

print("\nTop 10 classes with most non-food contamination:")
contamination_sorted = sorted(per_class_results, key=lambda x: x['nonfood_contamination_rate'], reverse=True)
for i, result in enumerate(contamination_sorted[:10], 1):
    if result['class'] == 'non_food':
        continue
    print(f"  {i}. {result['class']}: {result['nonfood_contamination_rate']:.2f}% ({result['nonfood_in_top3_count']}/{result['total_images']})")


126-CLASS MODEL INFERENCE RESULTS
Total images tested: 53918
Overall Top-1 Accuracy: 96.21% (51873/53918)
Overall Top-3 Accuracy (≥70% conf): 93.64% (50487/53918)

Bottom 10 classes by Top-1 Accuracy:
  1. pork_bistek: 74.86% (262/350)
  2. pritong_tilapia: 86.00% (301/350)
  3. beef_bistek: 87.43% (306/350)
  4. crispy_pata: 88.00% (308/350)
  5. chicken_adobo: 88.57% (310/350)
  6. steak: 88.86% (311/350)
  7. beef_kaldereta: 89.14% (312/350)
  8. pork_sisig: 89.14% (312/350)
  9. french_toast: 90.00% (315/350)
  10. pritong_galunggong: 90.29% (316/350)

Top 10 classes with most non-food contamination:
  1. orange: 59.14% (207/350)
  2. strawberry: 54.00% (189/350)
  3. apple: 52.57% (184/350)
  4. banana: 36.00% (126/350)
  5. biko: 14.00% (49/350)
  6. pritong_tilapia: 13.71% (48/350)
  7. white_rice: 11.71% (41/350)
  8. lumpiang_shanghai: 11.14% (39/350)
  9. guacamole: 10.86% (38/350)
  10. pritong_galunggong: 10.57% (37/350)


In [6]:
# Create output directory
output_dir = r'D:\VSC FILES\testtrain\inference_test_on_whole_dataset'
os.makedirs(output_dir, exist_ok=True)

# Save comprehensive text report
report_path = os.path.join(output_dir, 'inference_report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("="*100 + "\n")
    f.write("126-CLASS MODEL INFERENCE REPORT\n")
    f.write("="*100 + "\n\n")
    
    f.write("OVERALL SUMMARY\n")
    f.write("-"*100 + "\n")
    f.write(f"Total images tested: {total_images}\n")
    f.write(f"Total classes: {len(class_metrics)}\n\n")
    
    f.write(f"Overall Top-1 Accuracy: {overall_top1_acc:.2f}% ({overall_top1_correct}/{total_images})\n")
    f.write(f"Overall Top-3 Accuracy (≥70% confidence): {overall_top3_70conf_acc:.2f}% ({overall_top3_70conf_correct}/{total_images})\n")
    f.write("\n" + "="*100 + "\n\n")
    
    f.write("PER-CLASS DETAILED RESULTS\n")
    f.write("-"*100 + "\n")
    f.write(f"{'Class':<30} {'Total':>8} {'Top1':>8} {'Top1%':>8} {'Top3@70%':>10} {'T3@70%':>8} {'NF-Top3':>10} {'NF%':>8}\n")
    f.write("-"*100 + "\n")
    
    for result in sorted(per_class_results, key=lambda x: x['class']):
        f.write(f"{result['class']:<30} "
                f"{result['total_images']:>8} "
                f"{result['top1_correct']:>8} "
                f"{result['top1_accuracy']:>7.2f}% "
                f"{result['top3_70conf_correct']:>10} "
                f"{result['top3_70conf_accuracy']:>7.2f}% "
                f"{result['nonfood_in_top3_count']:>10} "
                f"{result['nonfood_contamination_rate']:>7.2f}%\n")
    
    f.write("\n" + "="*100 + "\n\n")
    
    f.write("BOTTOM 20 CLASSES BY TOP-1 ACCURACY\n")
    f.write("-"*100 + "\n")
    for i, result in enumerate(per_class_results_sorted[:20], 1):
        f.write(f"{i:2d}. {result['class']:<30} {result['top1_accuracy']:>6.2f}% ({result['top1_correct']}/{result['total_images']})\n")
    
    f.write("\n" + "="*100 + "\n\n")
    
    f.write("TOP 20 CLASSES WITH MOST NON-FOOD CONTAMINATION\n")
    f.write("-"*100 + "\n")
    count = 0
    for result in contamination_sorted:
        if result['class'] == 'non_food':
            continue
        count += 1
        f.write(f"{count:2d}. {result['class']:<30} {result['nonfood_contamination_rate']:>6.2f}% ({result['nonfood_in_top3_count']}/{result['total_images']})\n")
        if count >= 20:
            break
    
    f.write("\n" + "="*100 + "\n")
    f.write("\nLegend:\n")
    f.write("  Top1     = Number of images correctly predicted as top-1\n")
    f.write("  Top1%    = Percentage of Top-1 correct predictions\n")
    f.write("  Top3@70% = Number of images with correct class in top-3 with ≥70% confidence\n")
    f.write("  T3@70%   = Percentage of Top-3 at 70% confidence\n")
    f.write("  NF-Top3  = Number of times 'non_food' appeared in top-3 (for food classes only)\n")
    f.write("  NF%      = Non-food contamination rate\n")

print(f"Saved comprehensive text report to: {report_path}")

Saved comprehensive text report to: D:\VSC FILES\testtrain\inference_test_on_whole_dataset\inference_report.txt


In [7]:
# Save JSON with detailed per-class metrics
json_path = os.path.join(output_dir, 'per_class_metrics.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump({
        'overall_metrics': {
            'total_images': total_images,
            'top1_correct': overall_top1_correct,
            'top1_accuracy': overall_top1_acc,
            'top3_70conf_correct': overall_top3_70conf_correct,
            'top3_70conf_accuracy': overall_top3_70conf_acc
        },
        'per_class_results': per_class_results
    }, f, indent=2)

print(f"Saved JSON metrics to: {json_path}")

# Save summary statistics
summary_path = os.path.join(output_dir, 'summary.txt')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("QUICK SUMMARY\n")
    f.write("="*80 + "\n\n")
    f.write(f"Total Images: {total_images}\n")
    f.write(f"Total Classes: {len(class_metrics)}\n\n")
    f.write(f"Overall Top-1 Accuracy: {overall_top1_acc:.2f}%\n")
    f.write(f"Overall Top-3 at 70% Conf: {overall_top3_70conf_acc:.2f}%\n\n")
    
    # Calculate how many classes are in each performance tier
    excellent = sum(1 for r in per_class_results if r['top1_accuracy'] >= 90)
    good = sum(1 for r in per_class_results if 80 <= r['top1_accuracy'] < 90)
    fair = sum(1 for r in per_class_results if 70 <= r['top1_accuracy'] < 80)
    poor = sum(1 for r in per_class_results if 60 <= r['top1_accuracy'] < 70)
    very_poor = sum(1 for r in per_class_results if r['top1_accuracy'] < 60)
    
    f.write("Performance Distribution (by Top-1 Accuracy):\n")
    f.write(f"  Excellent (≥90%):  {excellent:3d} classes\n")
    f.write(f"  Good (80-89%):     {good:3d} classes\n")
    f.write(f"  Fair (70-79%):     {fair:3d} classes\n")
    f.write(f"  Poor (60-69%):     {poor:3d} classes\n")
    f.write(f"  Very Poor (<60%):  {very_poor:3d} classes\n")
    f.write("\n" + "="*80 + "\n")

print(f"Saved summary to: {summary_path}")

print(f"\n{'='*80}")
print(f"All reports saved to: {output_dir}")
print(f"{'='*80}")
print(f"\nFiles created:")
print(f"  1. inference_report.txt  - Comprehensive detailed report")
print(f"  2. per_class_metrics.json - Machine-readable metrics")
print(f"  3. summary.txt           - Quick summary statistics")

Saved JSON metrics to: D:\VSC FILES\testtrain\inference_test_on_whole_dataset\per_class_metrics.json
Saved summary to: D:\VSC FILES\testtrain\inference_test_on_whole_dataset\summary.txt

All reports saved to: D:\VSC FILES\testtrain\inference_test_on_whole_dataset

Files created:
  1. inference_report.txt  - Comprehensive detailed report
  2. per_class_metrics.json - Machine-readable metrics
  3. summary.txt           - Quick summary statistics
